In [30]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import norm
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed
import warnings

parquet_df = pd.read_parquet('../data/processed/iv_surface.parquet')
print(f'Loaded {len(parquet_df):,} rows')
print(parquet_df['split'].value_counts())

Loaded 4,445,206 rows
split
train            2651495
val               798788
test_temporal     718467
test_ticker       276456
Name: count, dtype: int64


## Baseline 2: SVI (Stochastic Volatility Inspired)

Raw SVI (Gatheral & Jacquier, 2014), fit per (ticker, date, maturityBucket) slice on total variance:

```
w(k) = a + b ( ρ(k − m) + √((k − m)² + σ²) )
```

Fit independently per slice — an in-sample calibration, not a prediction model. See report Methodology (Rung 5) for the full rationale and role in the ablation ladder.

In [31]:
def svi_total_variance(k, a, b, rho, m, sigma):
    """Raw SVI: total variance w(k) as a function of log-moneyness k."""
    return a + b * (rho * (k - m) + np.sqrt((k - m)**2 + sigma**2))


# Parameter bounds: b >= 0, |rho| < 1, sigma > 0; a and m unconstrained
SVI_BOUNDS = (
    [-np.inf, 0.0,      -1 + 1e-6, -np.inf, 1e-6  ],
    [ np.inf, np.inf,    1 - 1e-6,  np.inf, np.inf],
)


def fit_svi_slice(k, w):
    """
    Fit SVI to a single (ticker, tradingDate, maturityBucket) slice.

    Requires >= 5 data points (one per free parameter). Initialises 'a' near
    the minimum observed variance and 'm' at the log-moneyness of the
    minimum-variance contract. Returns (a, b, rho, m, sigma) or None if the
    slice is under-determined or the optimiser fails to converge.
    """
    if len(k) < 5:
        return None

    a0 = max(float(np.percentile(w, 5)), 1e-6)
    m0 = float(k[np.argmin(w)])

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            popt, _ = curve_fit(
                svi_total_variance, k, w,
                p0=[a0, 0.1, -0.5, m0, 0.1],
                bounds=SVI_BOUNDS,
                maxfev=5000,
            )
        return popt
    except (RuntimeError, ValueError):
        return None

### Fitting Approach

One slice = one (ticker, tradingDate, maturityBucket) triple, fit via `scipy.optimize.curve_fit` (Levenberg–Marquardt). Slices fit in parallel with `joblib`.

In [32]:
import time

SLICE_KEYS = ['ticker', 'tradingDate', 'maturityBucket']

# Collect slices as (key, k-array, w-array) tuples for parallel dispatch
slice_groups = [
    ((ticker, date, bucket),
     grp['logMoneyness'].values,
     grp['totalVariance'].values)
    for (ticker, date, bucket), grp in parquet_df.groupby(SLICE_KEYS)
]
print(f'Total slices to fit: {len(slice_groups):,}')


def _fit_one(key, k, w):
    return key, fit_svi_slice(k, w)


t0 = time.time()
raw_results = Parallel(n_jobs=-1, verbose=0)(
    delayed(_fit_one)(key, k, w) for key, k, w in slice_groups
)
elapsed = time.time() - t0

params_dict = {key: params for key, params in raw_results}
n_slices = len(params_dict)
n_failed = sum(p is None for p in params_dict.values())
print(f'Fitted {n_slices - n_failed:,}/{n_slices:,} slices in {elapsed:.1f}s  ({n_failed} failed to converge)')

Total slices to fit: 14,021
Fitted 12,024/14,021 slices in 521.0s  (1997 failed to converge)


In [33]:
# Convergence diagnostics: failure rate by ticker and maturityBucket
diag_rows = [
    {'ticker': t, 'maturityBucket': b, 'failed': params is None}
    for (t, _, b), params in params_dict.items()
]
diag_df = pd.DataFrame(diag_rows)

print('Failure rate by ticker:')
print(
    diag_df.groupby('ticker')['failed']
    .agg(failed='sum', total='count')
    .assign(pct=lambda x: (x['failed'] / x['total'] * 100).round(2))
    .to_string()
)

print('\nFailure rate by maturityBucket:')
print(
    diag_df.groupby('maturityBucket')['failed']
    .agg(failed='sum', total='count')
    .assign(pct=lambda x: (x['failed'] / x['total'] * 100).round(2))
    .to_string()
)

Failure rate by ticker:
        failed  total    pct
ticker                      
AAPL       370   1616  22.90
BA         449   1616  27.78
COST       195   1599  12.20
EOG        118   1584   7.45
GS         339   1616  20.98
JPM         52    284  18.31
LLY        165   1615  10.22
MSFT        17    284   5.99
NVDA        60   1615   3.72
SPY        214   1624  13.18
TSLA         0    284   0.00
XOM         18    284   6.34

Failure rate by maturityBucket:
                failed  total    pct
maturityBucket                      
14d                392   2019  19.42
180d               246   2020  12.18
1Y                  18   2020   0.89
30d                266   2020  13.17
60d                392   1931  20.30
7d                 345   2020  17.08
90d                338   1991  16.98


### Evaluation

SVI RMSE is a same-day fitting residual, not an out-of-sample prediction error (see report Methodology, Rung 5). Computed in volatility points for direct comparison to the polynomial and NN.

In [34]:
# Black-Scholes helpers (identical to polynomial.ipynb)
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def compute_pred_prices(df, pred_iv):
    S, K, T, r = df['uClose'].values, df['okey_xx'].values, df['years'].values, df['rate'].values
    is_call = df['callPut'].values == 1
    return np.where(
        is_call,
        black_scholes_call(S, K, T, r, pred_iv),
        black_scholes_put(S, K, T, r, pred_iv)
    )

def butterfly_violations(df, tol=1e-8):
    """
    Vectorized butterfly arbitrage check.
    Within each (ticker, date, maturity, callPut) group sorted by strike,
    the second derivative of option price w.r.t. strike must be >= 0.
    """
    grp = ['ticker', 'tradingDate', 'years', 'callPut']
    df = df.sort_values(grp + ['okey_xx'])
    p_prev = df.groupby(grp)['pred_price'].shift(1)
    p_next = df.groupby(grp)['pred_price'].shift(-1)
    k_prev = df.groupby(grp)['okey_xx'].shift(1)
    k_next = df.groupby(grp)['okey_xx'].shift(-1)
    h1 = df['okey_xx'] - k_prev
    h2 = k_next - df['okey_xx']
    valid = (h1 > 0) & (h2 > 0)
    second_deriv = (2.0 / (h1 + h2)) * (
        (p_next - df['pred_price']) / h2 - (df['pred_price'] - p_prev) / h1
    )
    checks     = int(valid.sum())
    violations = int((valid & (second_deriv < -tol)).sum())
    return violations / checks if checks > 0 else 0.0, checks

def calendar_violations(df, tol=1e-8):
    """
    Vectorized calendar spread arbitrage check.
    Within each (ticker, date, strike, callPut) group sorted by maturity,
    total implied variance must be non-decreasing in time to expiry.
    """
    grp = ['ticker', 'tradingDate', 'okey_xx', 'callPut']
    df = df.sort_values(grp + ['years'])
    tv_prev = df.groupby(grp)['pred_total_var'].shift(1)
    valid   = tv_prev.notna()
    checks     = int(valid.sum())
    violations = int((valid & (tv_prev > df['pred_total_var'] + tol)).sum())
    return violations / checks if checks > 0 else 0.0, checks

In [35]:
def apply_svi_predictions(df):
    """Look up fitted SVI params for each (ticker, date, bucket) and return predicted totalVariance."""
    result = pd.Series(np.nan, index=df.index, dtype=float)
    for (ticker, date, bucket), grp in df.groupby(SLICE_KEYS):
        params = params_dict.get((ticker, date, bucket))
        if params is None:
            continue
        result.loc[grp.index] = svi_total_variance(grp['logMoneyness'].values, *params)
    return result.values


def evaluate_svi_split(df, label):
    pred_tv   = apply_svi_predictions(df)
    valid     = ~np.isnan(pred_tv)
    df_eval   = df[valid].copy()
    pred_tv_v = pred_tv[valid]

    pred_midIV   = np.sqrt(np.maximum(pred_tv_v, 0) / df_eval['years'].values)
    actual_midIV = df_eval['midIV'].values
    rmse_vol = np.sqrt(mean_squared_error(actual_midIV, pred_midIV)) * 100

    df_eval['pred_price']     = compute_pred_prices(df_eval, pred_midIV)
    bf_rate, bf_checks        = butterfly_violations(df_eval)

    df_eval['pred_total_var'] = pred_tv_v
    cal_rate, cal_checks      = calendar_violations(df_eval)

    n_excl = int((~valid).sum())
    print(f'{label}')
    print(f'  Coverage  = {int(valid.sum()):,}/{len(df):,} rows  ({n_excl} from failed slices)')
    print(f'  RMSE      = {rmse_vol:.2f} vol pts')
    print(f'  Butterfly = {bf_rate:.4%} of {bf_checks:,} checks')
    print(f'  Calendar  = {cal_rate:.4%} of {cal_checks:,} checks')
    print()

    return {
        'split':             label,
        'rmse_volpts':       rmse_vol,
        'bf_violation_pct':  bf_rate  * 100,
        'cal_violation_pct': cal_rate * 100,
    }


svi_results = []
for split_name, label in [
    ('train',         'train         (Jan-Aug,  training tickers)'),
    ('val',           'val           (Sep-Oct,  training tickers)'),
    ('test_temporal', 'test_temporal (Nov-Dec,  training tickers)'),
    ('test_ticker',   'test_ticker   (Nov-Dec,  held-out tickers)'),
]:
    svi_results.append(evaluate_svi_split(parquet_df[parquet_df['split'] == split_name], label))

svi_results_df = pd.DataFrame(svi_results).set_index('split')
svi_results_df.round({'rmse_volpts': 2, 'bf_violation_pct': 4, 'cal_violation_pct': 4})

train         (Jan-Aug,  training tickers)
  Coverage  = 2,430,936/2,651,495 rows  (220559 from failed slices)
  RMSE      = 5.53 vol pts
  Butterfly = 0.0968% of 2,352,112 checks
  Calendar  = 0.0538% of 2,124,825 checks

val           (Sep-Oct,  training tickers)
  Coverage  = 742,358/798,788 rows  (56430 from failed slices)
  RMSE      = 4.99 vol pts
  Butterfly = 0.0412% of 718,444 checks
  Calendar  = 0.0568% of 645,868 checks

test_temporal (Nov-Dec,  training tickers)
  Coverage  = 641,447/718,467 rows  (77020 from failed slices)
  RMSE      = 4.67 vol pts
  Butterfly = 0.1670% of 619,585 checks
  Calendar  = 0.1116% of 551,743 checks

test_ticker   (Nov-Dec,  held-out tickers)
  Coverage  = 268,330/276,456 rows  (8126 from failed slices)
  RMSE      = 6.46 vol pts
  Butterfly = 0.0420% of 257,088 checks
  Calendar  = 0.1626% of 238,008 checks



,rmse_volpts,bf_violation_pct,cal_violation_pct
split,,,
"train (Jan-Aug, training tickers)",5.53,0.0968,0.0538
"val (Sep-Oct, training tickers)",4.99,0.0412,0.0568
"test_temporal (Nov-Dec, training tickers)",4.67,0.1670,0.1116
"test_ticker (Nov-Dec, held-out tickers)",6.46,0.0420,0.1626


In [36]:
from pathlib import Path

results_dir = Path("../results")
svi_results_df.to_csv(results_dir / "svi_test_results.csv")
print(f"Saved to {results_dir / 'svi_test_results.csv'}")

Saved to ../results/svi_test_results.csv


### Visualisations

Example fit, term structure across maturities, and RMSE comparison vs. the polynomial baseline.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../results/figures/svi')
figures_dir.mkdir(parents=True, exist_ok=True)
test_temp   = parquet_df[parquet_df['split'] == 'test_temporal']

# Figure 1: Example SVI fit — NVDA 30d slice with the best strike coverage in test_temporal
ex_ticker, ex_bucket = 'NVDA', '30d'
grp = test_temp[(test_temp['ticker'] == ex_ticker) & (test_temp['maturityBucket'] == ex_bucket)]

best_date = max(
    (d for d, dg in grp.groupby('tradingDate') if params_dict.get((ex_ticker, d, ex_bucket)) is not None),
    key=lambda d: len(grp[grp['tradingDate'] == d]),
    default=None,
)

slice_df  = grp[grp['tradingDate'] == best_date].sort_values('logMoneyness')
params    = params_dict[(ex_ticker, best_date, ex_bucket)]
years_avg = slice_df['years'].mean()

k_data = slice_df['logMoneyness'].values
pad    = (k_data.max() - k_data.min()) * 0.08
k_grid = np.linspace(k_data.min() - pad, k_data.max() + pad, 300)
iv_fit = np.sqrt(np.maximum(svi_total_variance(k_grid, *params), 0) / years_avg) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(slice_df['logMoneyness'], slice_df['midIV'] * 100,
           s=25, color='steelblue', zorder=5, label='Market quotes', alpha=0.85)
ax.plot(k_grid, iv_fit, color='tomato', linewidth=2, label='SVI fit')
ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Log-moneyness  k = log(K/S)')
ax.set_ylabel('Implied volatility (%)')
ax.set_title(f'SVI fit — {ex_ticker}  |  {pd.Timestamp(best_date).strftime("%Y-%m-%d")}  |  {ex_bucket} bucket')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / 'svi_example_fit.png', dpi=150)
plt.show()
print(f'Saved: {figures_dir}/svi_example_fit.png  ({len(slice_df)} contracts)')

In [ ]:
# Figure 2: SVI fitted smiles across all maturity buckets — term structure
# SPY on the date in test_temporal with the most converged bucket fits
from collections import Counter

test_temp = parquet_df[parquet_df['split'] == 'test_temporal']
spy_temp  = test_temp[test_temp['ticker'] == 'SPY']
BUCKET_ORDER = ['7d', '14d', '30d', '60d', '90d', '180d', '1Y']

spy_dates = set(spy_temp['tradingDate'].unique())  # restrict to test_temporal only
date_counts = Counter(
    date for (ticker, date, bucket), params in params_dict.items()
    if ticker == 'SPY' and params is not None and date in spy_dates
)
ts_date = max(date_counts, key=date_counts.get)

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(BUCKET_ORDER)))
fig, ax = plt.subplots(figsize=(8, 4.5))

for b, color in zip(BUCKET_ORDER, colors):
    key = ('SPY', ts_date, b)
    if params_dict.get(key) is None:
        continue
    sd = spy_temp[(spy_temp['tradingDate'] == ts_date) & (spy_temp['maturityBucket'] == b)]
    if len(sd) == 0:
        continue
    years_avg = sd['years'].mean()
    k_data = sd['logMoneyness'].values
    k_grid = np.linspace(k_data.min() - 0.02, k_data.max() + 0.02, 300)
    iv_fit = np.sqrt(np.maximum(svi_total_variance(k_grid, *params_dict[key]), 0) / years_avg) * 100
    ax.plot(k_grid, iv_fit, color=color, linewidth=1.8, label=b)

ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Log-moneyness  k = log(K/S)')
ax.set_ylabel('Implied volatility (%)')
ax.set_title(f'SVI fitted smiles by maturity — SPY  |  {pd.Timestamp(ts_date).strftime("%Y-%m-%d")}')
ax.legend(title='Bucket', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / 'svi_term_structure.png', dpi=150)
plt.show()
print(f'Saved: {figures_dir}/svi_term_structure.png  ({date_counts[ts_date]} buckets fitted, date={pd.Timestamp(ts_date).strftime("%Y-%m-%d")})')

### Comparison with Polynomial Baseline

SVI is evaluated in-sample; the polynomial is evaluated out-of-sample. See report Methodology and Results for interpretation.

In [39]:
# Side-by-side comparison: SVI (in-sample) vs polynomial (out-of-sample)
poly_csv = pd.read_csv('../results/poly_test_results.csv')

def _poly_row(prefix):
    match = poly_csv[poly_csv['split'].str.startswith(prefix)]
    return match.iloc[0] if len(match) > 0 else None

def _svi_row(prefix):
    match = svi_results_df[svi_results_df.index.str.startswith(prefix)]
    return match.iloc[0] if len(match) > 0 else None

records = []
for prefix, label in [('test_temporal', 'test_temporal'), ('test_ticker', 'test_ticker')]:
    pr = _poly_row(prefix)
    sr = _svi_row(prefix)
    if pr is not None:
        records.append({
            'Model': 'Polynomial (deg 5)',
            'Split': label,
            'Evaluation': 'out-of-sample prediction',
            'RMSE (vol pts)': round(float(pr['rmse_volpts']), 2),
            'Butterfly %': round(float(pr['bf_violation_pct']), 4),
            'Calendar %': round(float(pr['cal_violation_pct']), 4),
        })
    if sr is not None:
        records.append({
            'Model': 'SVI (per-slice)',
            'Split': label,
            'Evaluation': 'in-sample interpolation',
            'RMSE (vol pts)': round(float(sr['rmse_volpts']), 2),
            'Butterfly %': round(float(sr['bf_violation_pct']), 4),
            'Calendar %': round(float(sr['cal_violation_pct']), 4),
        })

pd.DataFrame(records).set_index(['Model', 'Split'])

,,Evaluation,RMSE (vol pts),Butterfly %,Calendar %
Model,Split,,,,
Polynomial (deg 5),test_temporal,out-of-sample prediction,11.74,0.0000,0.3080
SVI (per-slice),test_temporal,in-sample interpolation,4.67,0.1670,0.1116
Polynomial (deg 5),test_ticker,out-of-sample prediction,15.03,0.0257,0.4557
SVI (per-slice),test_ticker,in-sample interpolation,6.46,0.0420,0.1626


In [ ]:
# Figure 3: RMSE comparison bar chart — Polynomial (out-of-sample) vs SVI (in-sample)
poly_temporal = float(poly_csv[poly_csv['split'].str.startswith('test_temporal')].iloc[0]['rmse_volpts'])
poly_ticker   = float(poly_csv[poly_csv['split'].str.startswith('test_ticker')].iloc[0]['rmse_volpts'])
svi_temporal  = float(svi_results_df[svi_results_df.index.str.startswith('test_temporal')].iloc[0]['rmse_volpts'])
svi_ticker    = float(svi_results_df[svi_results_df.index.str.startswith('test_ticker')].iloc[0]['rmse_volpts'])

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, poly_val, svi_val, title in [
    (axes[0], poly_temporal, svi_temporal, 'test_temporal\n(trained tickers, Nov–Dec)'),
    (axes[1], poly_ticker,   svi_ticker,   'test_ticker\n(held-out tickers, Nov–Dec)'),
]:
    bars = ax.bar(
        ['Polynomial\n(predict)', 'SVI\n(interpolate)'],
        [poly_val, svi_val],
        color=['steelblue', 'tomato'], alpha=0.85, width=0.45,
    )
    for bar, val in zip(bars, [poly_val, svi_val]):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.2,
                f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('RMSE (vol pts)')
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, max(poly_val, svi_val) * 1.25)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Polynomial (out-of-sample prediction) vs SVI (in-sample interpolation)',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig(figures_dir / 'svi_vs_poly_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {figures_dir}/svi_vs_poly_rmse.png')

---
### Takeaways

See report Results and Robustness & Discussion for interpretation of these numbers.